# Fault Injection — Visualisation: Faulty vs Normal

This notebook visualises **injected faults alongside clean data** so you can see
exactly what the model would receive under each failure mode.

**Failure Mode 1 — Missing Modality (Sensor Dropout).** For each frame a
Bernoulli gate decides whether each sensor is alive:

```
m_RGB   ~ Bernoulli(1 - p_drop_RGB)     1 = alive, 0 = dropped
m_LiDAR ~ Bernoulli(1 - p_drop_LiDAR)
```

A dropped image becomes a black (zero) frame; a dropped point cloud becomes
empty (N = 0). Severity sweep: `p_drop in {0.00, 0.25, 0.50, 0.75, 1.00}`.

As new fault types are added (e.g. temporal misalignment), this notebook will
gain a section for each, all sharing the same clean-vs-corrupt layout.

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import glob
import numpy as np
import matplotlib.pyplot as plt

from src import (load_image, load_lidar, load_calib_griffin,
                 load_labels_for_frame, get_file_lists,
                 project_lidar_to_image, plot_fusion, CAT_COLORS)
from src.fault_injectors import MissingModalityInjector

plt.rcParams['figure.dpi'] = 120
print('Imports OK')

Imports OK


## Setup — load a frame to corrupt

In [3]:
DATASET_ROOT = '../datasets/griffin_50scenes_25m/griffin-release/griffin_50scenes_25m/griffin-release'
VEH = os.path.join(DATASET_ROOT, 'vehicle-side')
CALIB_DIR = os.path.join(VEH, 'calib')
files = get_file_lists(VEH)

CAMERA    = 'front'
FRAME_IDX = 700

image = load_image(sorted(glob.glob(os.path.join(VEH,'camera',CAMERA,'*.png')))[FRAME_IDX])
points = load_lidar(files['lidar_plys'][FRAME_IDX])
K, T_ego_to_sensor = load_calib_griffin(CALIB_DIR, camera=CAMERA)
h, w = image.shape[:2]

print(f'Clean sample: image {image.shape}, points {points.shape}')

Clean sample: image (1080, 1920, 3), points (2879, 4)


## Part 1 — A single dropped modality, side by side

The clearest way to understand the fault is to force each outcome and view it
next to the clean input. We use the severity extremes (`p_drop = 1.0` forces a
drop) so the effect is deterministic for illustration.

In [4]:
def show_clean_vs_corrupt(image, points, corrupt, K, T_ego_to_sensor, title=''):
    """Top row: clean image | clean LiDAR-on-image.
       Bottom row: corrupt image | corrupt LiDAR-on-image."""
    h, w = image.shape[:2]
    uvd_clean = project_lidar_to_image(points[:, :3], K, T_ego_to_sensor, h, w)
    cimg, cpts = corrupt['image'], corrupt['points']
    uvd_corr = (project_lidar_to_image(cpts[:, :3], K, T_ego_to_sensor, h, w)
                if len(cpts) > 0 else np.empty((0, 3)))

    fig, ax = plt.subplots(2, 2, figsize=(16, 9))
    fig.suptitle(title, fontsize=14)

    ax[0,0].imshow(image); ax[0,0].set_title('CLEAN image'); ax[0,0].axis('off')
    ax[0,1].imshow(image)
    if len(uvd_clean): ax[0,1].scatter(uvd_clean[:,0], uvd_clean[:,1], c=uvd_clean[:,2],
                                        cmap='jet_r', s=3, vmin=1, vmax=50)
    ax[0,1].set_title(f'CLEAN fusion  ({len(uvd_clean)} pts)'); ax[0,1].axis('off')

    ax[1,0].imshow(cimg)
    tag_img = 'DROPPED (black)' if corrupt['m_rgb']==0 else 'kept'
    ax[1,0].set_title(f'CORRUPT image  [{tag_img}]'); ax[1,0].axis('off')
    ax[1,1].imshow(cimg)
    if len(uvd_corr): ax[1,1].scatter(uvd_corr[:,0], uvd_corr[:,1], c=uvd_corr[:,2],
                                       cmap='jet_r', s=3, vmin=1, vmax=50)
    tag_pts = 'DROPPED (empty)' if corrupt['m_lidar']==0 else 'kept'
    ax[1,1].set_title(f'CORRUPT fusion  ({len(uvd_corr)} pts)  [LiDAR {tag_pts}]')
    ax[1,1].axis('off')

    plt.tight_layout(); plt.show()

In [5]:
# Force a LiDAR drop (p_drop_lidar = 1.0 guarantees it)
inj = MissingModalityInjector(p_drop_lidar=1.0, p_drop_rgb=0.0, seed=0)
corrupt = inj(image, points)
show_clean_vs_corrupt(image, points, corrupt, K, T_ego_to_sensor,
                      title='LiDAR dropped — model must rely on the camera alone')

In [6]:
# Force a camera drop (p_drop_rgb = 1.0 guarantees it)
inj = MissingModalityInjector(p_drop_rgb=1.0, p_drop_lidar=0.0, seed=0)
corrupt = inj(image, points)
show_clean_vs_corrupt(image, points, corrupt, K, T_ego_to_sensor,
                      title='Camera dropped — black frame, LiDAR survives')

## Part 2 — The Bernoulli dropout schedule

Before touching data, you can inspect *which* frames a given `p_drop` would drop
over a sequence. This reproduces the style of the 10-frame example table in the
spec. With a fixed seed the schedule is reproducible across runs.

In [7]:
def plot_schedule(p_drop_lidar, p_drop_rgb, n_frames=50, seed=0):
    inj = MissingModalityInjector(p_drop_rgb=p_drop_rgb, p_drop_lidar=p_drop_lidar, seed=seed)
    sch = inj.simulate_sequence(n_frames)
    fig, ax = plt.subplots(2, 1, figsize=(14, 3), sharex=True)
    for a, key, lbl, pd in [(ax[0],'m_rgb','Camera',p_drop_rgb),
                            (ax[1],'m_lidar','LiDAR',p_drop_lidar)]:
        alive = sch[key]
        a.imshow(alive[None,:], aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
        a.set_yticks([]); a.set_ylabel(f'{lbl}\np_drop={pd}', fontsize=9, rotation=0, labelpad=40, va='center')
        dropped = (alive==0).sum()
        a.set_title(f'{lbl}: {dropped}/{n_frames} frames dropped '
                    f'(green=alive, red=dropped)', fontsize=9)
    ax[1].set_xlabel('frame index')
    plt.tight_layout(); plt.show()

plot_schedule(p_drop_lidar=0.75, p_drop_rgb=0.25, n_frames=50, seed=0)

## Part 3 — Severity sweep

The spec sweeps `p_drop in {0.00, 0.25, 0.50, 0.75, 1.00}`. Here we show, over a
sequence, the fraction of frames where each modality survives at each severity.
At `p_drop = 0` nothing drops; at `p_drop = 1` everything drops; in between the
survival rate tracks `1 - p_drop`.

In [8]:
sweep = [0.00, 0.25, 0.50, 0.75, 1.00]
n_frames = 2000
empirical_alive = []
for p in sweep:
    inj = MissingModalityInjector(p_drop_lidar=p, seed=0)
    sch = inj.simulate_sequence(n_frames)
    empirical_alive.append(sch['m_lidar'].mean())

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(sweep, [1-p for p in sweep], 'k--', label='theoretical survival (1 - p_drop)')
ax.plot(sweep, empirical_alive, 'o-', color='#2196F3', markersize=9,
        label=f'empirical survival ({n_frames} frames)')
ax.set_xlabel('p_drop (LiDAR)'); ax.set_ylabel('fraction of frames LiDAR survives')
ax.set_title('Severity sweep: LiDAR survival vs drop probability')
ax.set_ylim(-0.05, 1.05); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [9]:
# Visual severity sweep: one frame, increasing LiDAR drop probability
# (showing how many points survive a per-point analogy is NOT the model here;
#  missing-modality drops the WHOLE cloud — so we show the binary outcome over
#  several seeds to convey the probability.)
fig, axes = plt.subplots(1, len(sweep), figsize=(18, 4))
for ax, p in zip(axes, sweep):
    # Sample 20 independent Bernoulli outcomes to show the drop FREQUENCY
    inj = MissingModalityInjector(p_drop_lidar=p, seed=0)
    sch = inj.simulate_sequence(20)
    alive_frac = sch['m_lidar'].mean()
    ax.imshow(sch['m_lidar'][None,:], aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
    ax.set_title(f'p_drop={p}\n{int(alive_frac*100)}% alive')
    ax.set_yticks([]); ax.set_xticks([])
plt.suptitle('LiDAR availability across 20 frames at each severity '
             '(green=alive, red=dropped)', fontsize=12)
plt.tight_layout(); plt.show()

## Part 4 — Independent per-sensor, per-frame dropout on a real sequence

In a real run both sensors are gated independently every frame. This shows a
short sequence with `p_drop_rgb=0.25, p_drop_lidar=0.5`, rendering the actual
corrupted fusion the model would receive at each step.

In [10]:
SEQ_START, SEQ_LEN = 700, 6
inj = MissingModalityInjector(p_drop_rgb=0.25, p_drop_lidar=0.5, seed=1)

fig, axes = plt.subplots(1, SEQ_LEN, figsize=(20, 4))
cam_imgs = sorted(glob.glob(os.path.join(VEH,'camera',CAMERA,'*.png')))
for ax, idx in zip(axes, range(SEQ_START, SEQ_START+SEQ_LEN)):
    img_i = load_image(cam_imgs[idx])
    pts_i = load_lidar(files['lidar_plys'][idx])
    out = inj(img_i, pts_i)
    ax.imshow(out['image'])
    if len(out['points']) > 0:
        uvd = project_lidar_to_image(out['points'][:,:3], K, T_ego_to_sensor, h, w)
        if len(uvd): ax.scatter(uvd[:,0], uvd[:,1], c=uvd[:,2], cmap='jet_r', s=2, vmin=1, vmax=50)
    state = f"RGB={'O' if out['m_rgb'] else 'X'} LiDAR={'O' if out['m_lidar'] else 'X'}"
    ax.set_title(f'frame {idx}\n{state}', fontsize=9); ax.axis('off')
plt.suptitle('Independent dropout over a sequence  (O=kept, X=dropped)  '
             'p_drop_rgb=0.25, p_drop_lidar=0.5', fontsize=12)
plt.tight_layout(); plt.show()

## Notes & next steps

- **Zero-fill vs mean-fill.** Dropped images are zeroed by default (spec). After
  network normalisation a zero pixel is not neutral; if you want a post-norm
  neutral input, construct the injector with `fill='mean', mean_value=<dataset
  mean>`. Empty (N=0) is the only representation for a dropped cloud.
- **Reproducibility.** The injector is seeded by default, so a severity sweep is
  repeatable. Pass `seed=None` for fresh randomness each run.
- **Next fault type:** Temporal Misalignment (pair current LiDAR with a stale
  image from frame k - delta_k). It will get its own module
  `src/fault_injectors/temporal_misalignment.py` and a new section in this
  notebook.

In [1]:
import sys, os
import glob, math
import numpy as np
import matplotlib
matplotlib.use('Agg') # Best for headless rendering to MP4
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.animation import FuncAnimation, FFMpegWriter
from PIL import Image as PILImage
from tqdm import tqdm
from IPython.display import HTML

# Adjust path to import the local 'src' module
sys.path.insert(0, os.path.abspath('..'))

from src import (load_lidar, load_pose_griffin, load_calib_griffin,
                 load_labels_for_frame, get_file_lists, project_lidar_to_image,
                 ann_to_ego_corners_bev, CAT_COLORS)
from src.fault_injectors import MissingModalityInjector

# ================= Configuration =================
DATASET_ROOT = '../datasets/griffin_50scenes_25m/griffin-release/griffin_50scenes_25m/griffin-release'
VEH   = os.path.join(DATASET_ROOT, 'vehicle-side')
DRONE = os.path.join(DATASET_ROOT, 'drone-side')

FRAME_START = 600
FRAME_END   = 650     # Adjust for sequence length
FPS         = 10
DOWNSAMPLE  = 2
OUTPUT_MP4  = 'temporal_fault_injection.mp4'

# Drop probabilities for visualization
P_DROP_RGB   = 0.25
P_DROP_LIDAR = 0.50
SEED         = 42
# =================================================

files = get_file_lists(VEH, DRONE)
VEH_CALIB = os.path.join(VEH, 'calib')

CAM_SPEC = {
    'vehicle_front': (VEH, 'front'), 'vehicle_back': (VEH, 'back'),
    'vehicle_left': (VEH, 'left'), 'vehicle_right': (VEH, 'right'),
    'drone_bottom': (DRONE, 'bottom'), 'drone_front': (DRONE, 'front'),
    'drone_back': (DRONE, 'back'), 'drone_left': (DRONE, 'left'),
    'drone_right': (DRONE, 'right'),
}

ACTIVE_CAMERAS = list(CAM_SPEC.keys())
OVERLAY_CAMERAS = [c for c in ACTIVE_CAMERAS if c.startswith('vehicle')]

# 1. Load Calibration and Image Paths
CALIB = {}
IMG_FILES = {}
for cam in ACTIVE_CAMERAS:
    root, sensor = CAM_SPEC[cam]
    IMG_FILES[cam] = sorted(glob.glob(os.path.join(root, 'camera', sensor, '*.png')))
    if cam in OVERLAY_CAMERAS:
        CALIB[cam] = load_calib_griffin(VEH_CALIB, sensor)

def load_image_ds(path, ds=1):
    img = PILImage.open(path).convert('RGB')
    if ds > 1:
        w, h = img.size; img = img.resize((w//ds, h//ds), PILImage.BILINEAR)
    return np.array(img)

N_FRAMES = FRAME_END - FRAME_START

# 2. Preload Original Data
DATA = {'images': {c: [] for c in ACTIVE_CAMERAS}, 'pts': [], 'anns': []}
print(f"Preloading {N_FRAMES} frames...")
for idx in tqdm(range(FRAME_START, FRAME_END)):
    DATA['pts'].append(load_lidar(files['lidar_plys'][idx]))
    DATA['anns'].append(load_labels_for_frame(VEH, files['pose_files'], idx))
    for cam in ACTIVE_CAMERAS:
        il = IMG_FILES[cam]
        # Append image or a black backup if index overflows
        DATA['images'][cam].append(
            load_image_ds(il[idx], DOWNSAMPLE) if idx < len(il) 
            else np.zeros((1080//DOWNSAMPLE, 1920//DOWNSAMPLE, 3), np.uint8)
        )

# 3. Simulate and Precompute Faults
inj = MissingModalityInjector(p_drop_rgb=P_DROP_RGB, p_drop_lidar=P_DROP_LIDAR, seed=SEED)
drop_schedule = inj.simulate_sequence(N_FRAMES)

FAULTY_DATA = {'images': {c: [] for c in ACTIVE_CAMERAS}, 'pts': []}
for li in range(N_FRAMES):
    m_rgb = drop_schedule['m_rgb'][li]
    m_lidar = drop_schedule['m_lidar'][li]
    
    # Faulty Points
    FAULTY_DATA['pts'].append(DATA['pts'][li] if m_lidar else np.empty((0, 4), np.float32))
        
    # Faulty Images
    for cam in ACTIVE_CAMERAS:
        if m_rgb:
            FAULTY_DATA['images'][cam].append(DATA['images'][cam][li])
        else:
            h, w = DATA['images'][cam][li].shape[:2]
            FAULTY_DATA['images'][cam].append(np.zeros((h, w, 3), dtype=np.uint8))

# 4. Precompute LiDAR Projections
PROJECTIONS = {'normal': {cam: [] for cam in OVERLAY_CAMERAS},
               'faulty': {cam: [] for cam in OVERLAY_CAMERAS}}

print("Precomputing LiDAR projections...")
for cam in OVERLAY_CAMERAS:
    K, T_ego_to_sensor = CALIB[cam]
    h, w = DATA['images'][cam][0].shape[:2]
    for li in range(N_FRAMES):
        # Project Normal
        uvd_n = project_lidar_to_image(DATA['pts'][li][:,:3], K, T_ego_to_sensor, h, w)
        PROJECTIONS['normal'][cam].append(uvd_n)
        
        # Project Faulty
        if drop_schedule['m_lidar'][li]:
            uvd_f = project_lidar_to_image(FAULTY_DATA['pts'][li][:,:3], K, T_ego_to_sensor, h, w)
        else:
            uvd_f = np.empty((0, 3))
        PROJECTIONS['faulty'][cam].append(uvd_f)

# 5. Set up the Figure Grid Layout (10 Rows, 2 Columns)
ROWS = 10
COLS = 2
fig = plt.figure(figsize=(12, 3.5 * ROWS), facecolor='#0d0d0d')
gs = gridspec.GridSpec(ROWS, COLS, figure=fig, hspace=0.15, wspace=0.05)

AXES = {'normal': {}, 'faulty': {}}
ROW_NAMES = ['bev'] + ACTIVE_CAMERAS

for c, col_key in enumerate(['normal', 'faulty']):
    for r, name in enumerate(ROW_NAMES):
        ax = fig.add_subplot(gs[r,c])
        AXES[col_key][name] = ax
        ax.set_facecolor('#0d0d0d')
        
        if name == 'bev':
            ax.tick_params(colors='#555', labelsize=6)
            for sp in ax.spines.values(): sp.set_edgecolor('#333')
            ax.set_xlabel('x fwd (m)', fontsize=6, color='#555')
            ax.set_ylabel('y left (m)', fontsize=6, color='#555')
            # Add BEV Legend
            handles = [mpatches.Patch(color=c, label=k) for k,c in CAT_COLORS.items()]
            ax.legend(handles=handles, loc='upper right', fontsize=5, framealpha=0.3, labelcolor='white', facecolor='#1a1a1a', edgecolor='#333')
            title_text = "Bird's-Eye View (LiDAR)"
        else:
            ax.axis('off')
            title_text = name.replace('_', ' ').title()
            
        prefix = "Normal: " if col_key == 'normal' else "Faulty: "
        ax.set_title(prefix + title_text, fontsize=10, color='#ccc', pad=6)

main_title = fig.text(0.5, 0.99, '', ha='center', va='top', fontsize=16, color='white', fontfamily='monospace', weight='bold')

# 6. Initialize Plot Artists
IM_ARTISTS = {'normal': {}, 'faulty': {}}
SC_ARTISTS = {'normal': {}, 'faulty': {}}
BEV_SC = {}
ego_markers = {}
bev_box_patches = {'normal': [], 'faulty': []}
BEV_RANGE = 60

for col_key in ['normal', 'faulty']:
    for name, ax in AXES[col_key].items():
        if name == 'bev':
            BEV_SC[col_key] = ax.scatter([], [], c=[], cmap='plasma', s=0.5, alpha=0.7, vmin=-2, vmax=5)
            ego_markers[col_key], = ax.plot([0], [0], 'w^', markersize=8, zorder=10)
            ax.set_xlim(-BEV_RANGE, BEV_RANGE); ax.set_ylim(-BEV_RANGE, BEV_RANGE); ax.set_aspect('equal')
        else:
            h, w = DATA['images'][name][0].shape[:2]
            IM_ARTISTS[col_key][name] = ax.imshow(np.zeros((h, w, 3), np.uint8), aspect='auto', interpolation='bilinear')
            if name in OVERLAY_CAMERAS:
                SC_ARTISTS[col_key][name] = ax.scatter([], [], c=[], cmap='jet_r', s=1.5, alpha=0.85, vmin=1, vmax=50)

# 7. Animation Update Loop
def update(li):
    abs_frame = FRAME_START + li
    
    # Header Update
    rgb_state = 'ALIVE' if drop_schedule['m_rgb'][li] else 'DROPPED'
    lidar_state = 'ALIVE' if drop_schedule['m_lidar'][li] else 'DROPPED'
    state_color = 'white' if (drop_schedule['m_rgb'][li] and drop_schedule['m_lidar'][li]) else '#ff4444'
    main_title.set_text(f'Frame {abs_frame:04d} | Fault State: RGB={rgb_state}, LiDAR={lidar_state}')
    main_title.set_color(state_color)
    
    arts = []
    
    # Iterate over both sets of columns
    for col_key, dt_images, dt_pts, p_proj in [
        ('normal', DATA['images'], DATA['pts'], PROJECTIONS['normal']),
        ('faulty', FAULTY_DATA['images'], FAULTY_DATA['pts'], PROJECTIONS['faulty'])
    ]:
        # Cameras
        for name in ACTIVE_CAMERAS:
            IM_ARTISTS[col_key][name].set_data(dt_images[name][li])
            arts.append(IM_ARTISTS[col_key][name])
            if name in OVERLAY_CAMERAS:
                uvd = p_proj[name][li]
                if len(uvd) > 0:
                    SC_ARTISTS[col_key][name].set_offsets(uvd[:,:2]); SC_ARTISTS[col_key][name].set_array(uvd[:,2])
                else:
                    SC_ARTISTS[col_key][name].set_offsets(np.empty((0,2))); SC_ARTISTS[col_key][name].set_array(np.array([]))
                arts.append(SC_ARTISTS[col_key][name])
                
        # BEV Update
        pts = dt_pts[li]
        if len(pts) > 0:
            BEV_SC[col_key].set_offsets(pts[:,:2]); BEV_SC[col_key].set_array(pts[:,2])
        else:
            BEV_SC[col_key].set_offsets(np.empty((0,2))); BEV_SC[col_key].set_array(np.array([]))
        arts.extend([BEV_SC[col_key], ego_markers[col_key]])
        
        # Bounding Boxes
        for p in bev_box_patches[col_key]: p.remove()
        bev_box_patches[col_key] = []
        for ann in DATA['anns'][li]:
            c = CAT_COLORS.get(ann.get('category', 'car').lower(), '#fff')
            poly = plt.Polygon(ann_to_ego_corners_bev(ann), fill=False, edgecolor=c, linewidth=1.2, alpha=0.9)
            AXES[col_key]['bev'].add_patch(poly)
            bev_box_patches[col_key].append(poly)
            
    return arts

# 8. Render and Save File
print("\nAnimating and saving MP4...")
plt.tight_layout(rect=[0, 0, 1, 0.98])
anim = FuncAnimation(fig, update, frames=N_FRAMES, interval=1000//FPS, blit=False, repeat=True)

writer = FFMpegWriter(fps=FPS, bitrate=4000)
anim.save(OUTPUT_MP4, writer=writer, dpi=100, 
          progress_callback=lambda i, n: print(f'  Rendering Frame {i+1}/{n}', end='\r'))

print(f"\nSaved animation successfully to {OUTPUT_MP4}!")
# HTML(anim.to_jshtml(fps=FPS)) # Uncomment to render inline in Jupyter instead

Preloading 50 frames...


100%|██████████| 50/50 [02:37<00:00,  3.15s/it]


Precomputing LiDAR projections...

Animating and saving MP4...


/tmp/ipykernel_2299/4076240427.py:228: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 1, 0.98])


  Rendering Frame 50/50
Saved animation successfully to temporal_fault_injection.mp4!
